<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Theoretical Foundations

The laboratory relies on the pinhole-camera model, planar projective geometry, normalized Direct Linear Transform (DLT), Singular Value Decomposition (SVD), and Zhang's planar calibration method.

## Mathematical Model

For world point $\mathbf{X}=[X,Y,Z,1]^T$ and image point $\mathbf{x}=[u,v,1]^T$,

$$
s\mathbf{x}=K\begin{bmatrix}R&t\end{bmatrix}\mathbf{X}.
$$

The intrinsic matrix is

$$
K=
\begin{bmatrix}
\alpha & \gamma & u_0\\
0 & \beta & v_0\\
0 & 0 & 1
\end{bmatrix},
$$

where $\alpha,\beta$ are focal scale factors, $\gamma$ is skew, and $(u_0,v_0)$ is the principal point.

## Planar Homography

For the calibration plane $Z=0$,

$$
s\mathbf{x}
=
K
\begin{bmatrix}
r_1&r_2&t
\end{bmatrix}
\begin{bmatrix}
X\\Y\\1
\end{bmatrix}
=
H\mathbf{X}_p.
$$

Thus

$$
H=K\begin{bmatrix}r_1&r_2&t\end{bmatrix}.
$$

Each valid view provides one plane-to-image homography.

## Point Normalization

Before DLT, each 2D point set is translated so its centroid is at the origin and scaled so its mean distance from the origin is $\sqrt{2}$.

If $\bar d$ is the mean distance,

$$
s_n=\frac{\sqrt{2}}{\bar d},
$$

and

$$
T=
\begin{bmatrix}
s_n&0&-s_n\bar x\\
0&s_n&-s_n\bar y\\
0&0&1
\end{bmatrix}.
$$

Normalization improves the conditioning of the homogeneous homography system.

## Normalized DLT

Each point correspondence contributes two equations to

$$
Qh=0.
$$

With

$$
Q=U\Sigma V^T,
$$

the last right singular vector gives the homogeneous least-squares solution for the normalized homography. Denormalization gives

$$
H=T_{image}^{-1}H_nT_{plane}.
$$

## Zhang Intrinsic Constraints

Let

$$
H=[h_1\;h_2\;h_3],
\qquad
B=K^{-T}K^{-1}.
$$

Rotation-column orthogonality and equal norm give

$$
h_1^TBh_2=0,
$$

$$
h_1^TBh_1-h_2^TBh_2=0.
$$

For homography columns $h_i,h_j$, the implementation constructs

$$
v_{ij}=
\begin{bmatrix}
h_{i1}h_{j1}\\
h_{i1}h_{j2}+h_{i2}h_{j1}\\
h_{i2}h_{j2}\\
h_{i3}h_{j1}+h_{i1}h_{j3}\\
h_{i3}h_{j2}+h_{i2}h_{j3}\\
h_{i3}h_{j3}
\end{bmatrix}.
$$

All views are stacked into

$$
Vb=0,
$$

which is solved by SVD.

## Recovering the Intrinsic Matrix

Write

$$
b=[b_{11},b_{12},b_{22},b_{13},b_{23},b_{33}]^T.
$$

Define

$$
d=b_{11}b_{22}-b_{12}^2,
$$

$$
v_0=\frac{b_{12}b_{13}-b_{11}b_{23}}{d},
$$

$$
\lambda=
b_{33}
-
\frac{b_{13}^2+v_0(b_{12}b_{13}-b_{11}b_{23})}{b_{11}}.
$$

Then

$$
\alpha=\sqrt{\frac{\lambda}{b_{11}}},
\qquad
\beta=\sqrt{\frac{\lambda b_{11}}{d}},
$$

$$
\gamma=-\frac{b_{12}\alpha^2\beta}{\lambda},
\qquad
u_0=\frac{\gamma v_0}{\beta}-\frac{b_{13}\alpha^2}{\lambda}.
$$

These values form $K$.

## Camera Pose Recovery

For $H=[h_1\;h_2\;h_3]$,

$$
\lambda_p=\frac{1}{\lVert K^{-1}h_1\rVert},
$$

$$
r_1=\lambda_pK^{-1}h_1,
\qquad
r_2=\lambda_pK^{-1}h_2,
$$

$$
r_3=r_1\times r_2,
\qquad
t=\lambda_pK^{-1}h_3.
$$

The approximate rotation is projected onto the nearest valid rotation matrix with SVD and constrained to determinant $+1$.

## Reprojection Error

For measured point $\mathbf{x}_i$ and predicted point $\hat{\mathbf{x}}_i$,

$$
e_i=\lVert\hat{\mathbf{x}}_i-\mathbf{x}_i\rVert_2.
$$

The implementation reports

$$
\bar e=\frac{1}{N}\sum_i e_i,
$$

and

$$
\mathrm{RMSE}=\sqrt{\frac{1}{N}\sum_i e_i^2}.
$$

## Assumptions

- Planar calibration target.
- Known chessboard geometry.
- Correct corner correspondences.
- Sufficient geometric diversity among views.
- No radial or tangential distortion model.

## Interpretation

Calibration quality is evaluated through detected corners, estimated camera poses, measured-versus-reprojected overlays, per-view residuals, and the global reprojection-error distribution.

## Limitations of the Theory

The implemented pinhole model does not estimate radial or tangential lens distortion. Significant distortion may therefore appear as systematic reprojection residuals.

## Key Takeaways

- A planar view induces a homography.
- Multiple homographies constrain one shared intrinsic matrix.
- Point normalization improves DLT conditioning.
- SVD solves the homogeneous geometric systems.
- Intrinsics enable pose recovery.
- Reprojection error validates image-space geometric consistency.